# `core`
> set of functions and classes used across this package and usable for other packages

In [ ]:
#|default_exp core

In [ ]:
#| export
from __future__ import annotations
import configparser
import itertools
import json
import logging
import os
import sys
import warnings
from functools import wraps
from pathlib import Path
from typing import Any, Optional

import numpy as np

# Try to load Google drive package for Google Colab
try: from google.colab import drive  # type: ignore
except: pass

In [ ]:
#| hide
from nbdev import show_doc, nbdev_export

In [ ]:
#| export
# Retrieve the package root
from eccore import __file__
CODE_ROOT = Path(__file__).parents[0]
PACKAGE_ROOT = Path(__file__).parents[1]

In [ ]:
#| hide
CODE_ROOT, PACKAGE_ROOT

(Path('/home/vtec/projects/ec-packages/eccore/eccore'),
 Path('/home/vtec/projects/ec-packages/eccore'))

# Data structures

Classes to handle data structure more easily

In [ ]:
with open('data-dev/jsondict-test.json', 'r') as fp:
    d =  json.load(fp)
d

d.items()

dict_items([('b', 2), ('c', 3), ('d', 4)])

In [ ]:
# | export
class JsonDict(dict):
    """Dictionary whose current value is mirrored in a json file and can be initated from a json file
    
    `JsonDict` requires a path to json file at creation. An optional dict can be passed as argument.

    Behavior at creation:
    
    - `JsonDict(p2json, dict)` will create a `JsonDict` with key-values from `dict`, and mirrored in `p2json`
    - `JsonDict(p2json)` will create a `JsonDict` with empty dictionary and load json content if file exists

    Once created, `JsonDict` instances behave exactly as a dictionary
    """
    def __init__(
        self, 
        p2json: str|Path,       # path to the json file to mirror with the dictionary 
        dictionary: Optional[dict] = None  # optional dictionary to initialize the JsonDict
        ):
        """Create dict from a passed dict or from json. Create the json file if required"""
        self.p2json = Path(p2json) if isinstance(p2json, str) else p2json
        if dictionary is None:
            if self.p2json.is_file():
                dictionary = self.load()
                self.initial_dict_from_json = True
            else:
                dictionary = {}
                self.initial_dict_from_json = False
        super().__init__(dictionary.items()) # type: ignore
        self.save()
    
    def __setitem__(self, __k:Any, v:Any) -> None:
        super().__setitem__(__k, v)
        self.save()

    def __delitem__(self, k:Any):
        super().__delitem__(k)
        self.save()

    def __repr__(self):
        txt1 = super().__repr__()
        txt2 = f"\ndict mirrored in {self.p2json.absolute()}"
        return txt1 + txt2

    def load(self):
        with open(self.p2json, 'r') as fp:
            return json.load(fp)

    def save(self):
        with open(self.p2json, 'w') as fp:
            json.dump(self, fp, indent=4)



Create a new dictionary mirrored to a JSON file:

In [ ]:
d = {'a': 1, 'b': 2, 'c': 3}
p2json = Path('data-dev/jsondict-test.json')
jsond = JsonDict(p2json, d)
jsond

{'a': 1, 'b': 2, 'c': 3}
dict mirrored in /home/vtec/projects/ec-packages/eccore/nbs-dev/data-dev/jsondict-test.json

Once created, the `JsonFile` instance behaves exactly like a dictionary, with the added benefit that any change to the dictionary is automatically saved to the JSON file.

In [ ]:
jsond['a'], jsond['b'], jsond['c']

(1, 2, 3)

In [ ]:
for k, v in jsond.items():
    print(f"key: {k}; value: {v}")

key: a; value: 1
key: b; value: 2
key: c; value: 3


Adding or removing a value from the dictionary works in the same way as for a normal dictionary. But the json file is automatically updated.

In [ ]:
jsond['d'] = 4
jsond

{'a': 1, 'b': 2, 'c': 3, 'd': 4}
dict mirrored in /home/vtec/projects/ec-packages/eccore/nbs-dev/data-dev/jsondict-test.json

In [ ]:
with open(p2json, 'r') as fp:
    print(fp.read())

{
    "a": 1,
    "b": 2,
    "c": 3,
    "d": 4
}


In [ ]:
del jsond['a']
jsond

{'b': 2, 'c': 3, 'd': 4}
dict mirrored in /home/vtec/projects/ec-packages/eccore/nbs-dev/data-dev/jsondict-test.json

In [ ]:
with open(p2json, 'r') as fp:
    print(fp.read())

{
    "b": 2,
    "c": 3,
    "d": 4
}


# Validation functions

In [ ]:
#| export
def is_type(
    obj:Any,                 # object whose type to validate
    obj_type:type,           # expected type for `obj`
    raise_error:bool=False,  # when True, raise a ValueError is `obj` is not of the right type
)-> bool:                    # True when `obj` is of the right type, False otherwise 
    """Validate that `obj` is of type `obj_type`. Raise error in the negative when `raise_error` is `True`"""
    if not isinstance(obj_type, type): raise ValueError(f"{obj_type} is not a type")
    if isinstance(obj, obj_type): return True
    else:
        if raise_error: raise ValueError(f"passed object is not of type {obj_type}")
        else: return False

In [ ]:
show_doc(is_type)

---

[source](https://github.com/vtecftwy/eccore/blob/main/eccore/core.py#L88){target="_blank" style="float:right; font-size:smaller"}

### is_type

```python
def is_type(
    obj:Any, # object whose type to validate
    obj_type:type, # expected type for `obj`
    raise_error:bool=False, # when True, raise a ValueError is `obj` is not of the right type
)->bool: # True when `obj` is of the right type, False otherwise
```

*Validate that `obj` is of type `obj_type`. Raise error in the negative when `raise_error` is `True`*

In [ ]:
is_type(obj='this is a string', obj_type=str)

True

In [ ]:
is_type(obj=np.ones(shape=(2,2)), obj_type=np.ndarray)

True

## Path validation

Functions to ensure path are properly formated and point to a real file or directory.

In [ ]:
#| export
def validate_path(
    path:str|Path,           # path to validate
    path_type:str='file',    # type of the target path: `'file'`, `'dir'` or `'any'`
    raise_error:bool=False,  # when True, raise a ValueError is path does not point to a real file or directory
)-> bool:                    # True when path is a valid path, False otherwise 
    """Validate that path is a `Path` or `str` and points to a real file or directory"""
    if isinstance(path, str): 
        path = Path(path)
    if (path_type=='file' and path.is_file()) or (path_type=='dir' and path.is_dir()) :
        return True
    if path_type=='any' and path.exists():
        return True
    else:
        if raise_error: raise ValueError(f"No {'file or directory' if path_type=='any' else path_type} at {path.absolute()}. Check the path")
        else: return False

In [ ]:
show_doc(validate_path)

---

[source](https://github.com/vtecftwy/eccore/blob/main/eccore/core.py#L101){target="_blank" style="float:right; font-size:smaller"}

### validate_path

```python
def validate_path(
    path:str | Path, # path to validate
    path_type:str='file', # type of the target path: `'file'`, `'dir'` or `'any'`
    raise_error:bool=False, # when True, raise a ValueError is path does not point to a real file or directory
)->bool: # True when path is a valid path, False otherwise
```

*Validate that path is a `Path` or `str` and points to a real file or directory*

In [ ]:
path_file = Path('data-dev/jsondict-test.json')
validate_path(path_file)

True

In [ ]:
validate_path(path_file, path_type='any')

True

In [ ]:
path_dir = Path('../data')
validate_path(path_dir, path_type='dir')

True

In [ ]:
validate_path(path_dir, path_type='any')

True

In [ ]:
path_error = Path('../data/img/IIIMG_001_512px.jpg')
validate_path(path_error)

False

In [ ]:
path_error = Path('../data/img/IIIMG_001_512px.jpg')
try:
    validate_path(path_error, raise_error=True)
except ValueError as e:
    print('Raised an ValueError exception with error message:')
    print(e)

Raised an ValueError exception with error message:
No file at /home/vtec/projects/ec-packages/eccore/nbs-dev/../data/img/IIIMG_001_512px.jpg. Check the path


In [ ]:
#| export
def safe_path(
    path:str|Path, # path to validate
)-> Path:          # validated path returned as a pathlib.Path
    """Return a `Path` object when given a valid path as a `str` or a `Path`, raise error otherwise
    """
    validate_path(path, path_type='any', raise_error=True)
    if isinstance(path, str): 
        path = Path(path)
    return path

In [ ]:
show_doc(safe_path)

---

[source](https://github.com/vtecftwy/eccore/blob/main/eccore/core.py#L118){target="_blank" style="float:right; font-size:smaller"}

### safe_path

```python
def safe_path(
    path:str | Path, # path to validate
)->Path: # validated path returned as a pathlib.Path
```

*Return a `Path` object when given a valid path as a `str` or a `Path`, raise error otherwise*

In [ ]:
fname_with_file = 'data-dev/jsondict-test.json'
safe_path(fname_with_file)

Path('data-dev/jsondict-test.json')

In [ ]:
fname_without_file = 'data-dev/no-file-here.json'
try:
    safe_path(fname_without_file)
except ValueError as e:
    print('Raised an ValueError exception with error message:')
    print(e)

Raised an ValueError exception with error message:
No file or directory at /home/vtec/projects/ec-packages/eccore/nbs-dev/data-dev/no-file-here.json. Check the path


# Access key files and directories

In [ ]:
#| export

# TODO: consider how to modify this with fastcore's Config

def get_config_value(section:str,                                   # section in the configparser cfg file
                     key:str,                                       # key in the selected section
                     path_to_config_file:Optional[Path|str] = None  # path to the cfg file
                    )-> Any :                            # the value corresponding to section>key>value 
    """Returns the value corresponding to the key-value pair in the configuration file (configparser format)
    
    When no path_to_config_file is provided, the function will try to find the file in: the system's `home`, 
    the parent directory of the current directory, and the Google drive directory mounted to the Colab environment.
    """
    if path_to_config_file is None:
        # try several possible file locations
        possible_fnames = ['config-api-keys.cfg', 'config-sample.cfg']
        possible_dirs = [
            Path('').resolve(),         # current working directory
            Path('..').resolve(),       # current working directory's parent directory
            Path().home()/'.eccore',    # config file in local eccore config directory (home/.eccore/)
            Path('/content/gdrive/MyDrive/private-across-accounts/'), # google drive shared secret folder
            ] 
        # Cheach each possible position in the right order
        for directory, fname in itertools.product(possible_dirs, possible_fnames):
            if (directory/fname).is_file():
                path_to_config_file = directory/fname
                print('found', path_to_config_file)
                break
        # Raise exception if no file is found
        if path_to_config_file is None:
            raise ValueError(f"No config file found in possible_paths. Please provide a specific path")

    path_to_config_file = safe_path(path_to_config_file)
    print(f"Using config file at {path_to_config_file.absolute()}")
    configuration = configparser.ConfigParser()
    configuration.read(path_to_config_file)
    return configuration[section][key]

In [ ]:
show_doc(get_config_value)

---

[source](https://github.com/vtecftwy/eccore/blob/main/eccore/core.py#L133){target="_blank" style="float:right; font-size:smaller"}

### get_config_value

```python
def get_config_value(
    section:str, # section in the configparser cfg file
    key:str, # key in the selected section
    path_to_config_file:Optional[Path | str]=None, # path to the cfg file
)->Any: # the value corresponding to section>key>value
```

*Returns the value corresponding to the key-value pair in the configuration file (configparser format)*

When no path_to_config_file is provided, the function will try to find the file in: the system's `home`, 
the parent directory of the current directory, and the Google drive directory mounted to the Colab environment.

By defaults (`path_to_config_file is None`), it is assumed that the configuration file is located in:
- the local package config directory (home/.eccore/)
- the working directory
- the folder above the working directory
- the `private-accross-accounts directory` on google drive. 

File names are expected to be either `config-api-keys.cfg` or `config-sample.cfg`.

If not, a path to the file (`Path` or `str`) must be provided.

The configuration file is expected to be in the format used by the standard module `configparser` [documentation](https://docs.python.org/3/library/configparser.html)

```ascii
    [DEFAULT]
    key = value

    [section_name]
    key = value

    [section_name]
    key = value
```

In [ ]:
get_config_value(section="github", key="git_name")

found /home/vtec/projects/ec-packages/eccore/config-api-keys.cfg
Using config file at /home/vtec/projects/ec-packages/eccore/config-api-keys.cfg


'Etienne Charlier'

In [ ]:
path2cfg = Path('../config-sample.cfg').resolve()
assert path2cfg.is_file(), f"{path2cfg} is not a file"
print(path2cfg.absolute())

with open(path2cfg, 'r') as fp:
    print(fp.read())

/home/vtec/projects/ec-packages/eccore/config-sample.cfg
[azure]
azure-api-key= dummy_api_key_for_azure

[github]
git_name = not_my_real_github_name
git_email = not_my_real_git_email
github_username = not_my_real_git_username

[kaggle]
kaggle_username = not_my_real_kaggle_name
kaggle_key = dummy_api_key_for_kaggle

[wandb]
api_key = dummy_api_key_for_wandb



In [ ]:
value = get_config_value(section='azure', key='azure-api-key', path_to_config_file=path2cfg)
assert value == 'dummy_api_key_for_azure'

Using config file at /home/vtec/projects/ec-packages/eccore/config-sample.cfg


In [ ]:
value = get_config_value(section='kaggle', key='kaggle_username', path_to_config_file=path2cfg)
assert value == 'not_my_real_kaggle_name'

Using config file at /home/vtec/projects/ec-packages/eccore/config-sample.cfg


In [ ]:
value = get_config_value(section='wandb', key='api_key', path_to_config_file=path2cfg)
assert value == 'dummy_api_key_for_wandb'

Using config file at /home/vtec/projects/ec-packages/eccore/config-sample.cfg


In [ ]:
#|eval:  false
value = get_config_value(section='dummy', key='dummy-user-id')
assert value.startswith('dummy-userID-from')

found /home/vtec/projects/ec-packages/eccore/config-api-keys.cfg
Using config file at /home/vtec/projects/ec-packages/eccore/config-api-keys.cfg


# Setup utilities

In [ ]:
#| export
class CurrentMachine:
    """Callable class representing the current machine. When called, instance return a dict all `attrs`:
    
    - `os`: the operating system running on the machine
    - `home`: path to home on the machine
    - `is_local`, `is_colab`, `is_kaggle`: whether the machine is running locally or not
    - `p2config`: path to the config file
    - `package_root`: path to the package root directory

    CurrentMachine is a singleton class.
    """
    
    _instance = None
    _config_dir: str = '.ecutilities'
    _config_fname = 'ecutilities.cfg'

    def __new__(cls, *args, **kwargs):
        # Create instance if it does not exist yet
        if cls._instance is None:
            cls.home = Path.home().resolve()
            cls.p2config = cls.home / cls._config_dir / cls._config_fname
            cls.package_root = Path(__file__).parents[1]
            cls._instance = super().__new__(cls)
        return cls._instance
    
    def __init__(
        self, 
        mount_gdrive:bool=True  # True to mount Google Drive if running on Colab
        ):
            self.is_colab = 'google.colab' in sys.modules       
            if self.is_colab and mount_gdrive:
                drive.mount('/content/gdrive')
                self.gdrive = Path('/content/gdrive/MyDrive')

            self.is_kaggle = 'kaggle_web_client' in sys.modules
            if self.is_kaggle:
                raise NotImplementedError(f"ProjectFileSystem is not implemented for Kaggle yet")

            if not self.is_colab and not self.is_kaggle and not self.is_local:
                msg = """
                      Code does not seem to run on the cloud but computer is not registered as local
                      If you are running on a local computer, you must register it as local by running
                        `ProjectFileSystem().register_as_local()`
                      before you can use the ProjectFileSystem class.
                      """
                warnings.warn(msg, UserWarning)

    def __call__(self): 
        attrs = 'os home is_local is_colab is_kaggle p2config package_root'.split()
        d = {k: getattr(self, k,None) for k in attrs}
        return d

    def read_config(self):
        """Read config from the configuration file if it exists and return an empty config in does not"""
        cfg = configparser.ConfigParser()
        if self.p2config.is_file(): 
            cfg.read(self.p2config)
        else:
            cfg.add_section('Infra')
        return cfg
    
    def register_as_local(self):
        """Update the configuration file to register the machine as local machine"""
        cfg = self.read_config()
        os.makedirs(self.home/self._config_dir, exist_ok=True)
        cfg['Infra']['registered_as_local'] = 'True'
        with open(self.p2config, 'w') as fp:
            cfg.write(fp)
        return cfg
  
    def deregister_as_local(self):
        """Update the configuration file to deregister the machine from local machine status"""
        cfg = self.read_config()
        os.makedirs(self.home/self._config_dir, exist_ok=True)
        cfg['Infra']['registered_as_local'] = 'False'
        with open(self.p2config, 'w') as fp:
            cfg.write(fp)
        return cfg

    @property
    def home(self) -> Path: return Path.home().absolute()
    
    @property
    def os(self) : return sys.platform

    @property
    def p2config(self) -> Path: return self.home / self._config_dir / self._config_fname
           
    @property
    def is_local(self):
        """Return `True` if the current machine was registered as a local machine"""
        cfg = self.read_config()
        return cfg['Infra'].getboolean('registered_as_local', False)


In [ ]:
show_doc(CurrentMachine)

---

[source](https://github.com/vtecftwy/eccore/blob/main/eccore/core.py#L168){target="_blank" style="float:right; font-size:smaller"}

### CurrentMachine

```python
def CurrentMachine(
    mount_gdrive:bool=True, # True to mount Google Drive if running on Colab
):
```

*Callable class representing the current machine. When called, instance return a dict all `attrs`:*

- `os`: the operating system running on the machine
- `home`: path to home on the machine
- `is_local`, `is_colab`, `is_kaggle`: whether the machine is running locally or not
- `p2config`: path to the config file
- `package_root`: path to the package root directory

CurrentMachine is a singleton class.

In [ ]:
machine = CurrentMachine()
machine()

{'os': 'linux',
 'home': Path('/home/vtec'),
 'is_local': True,
 'is_colab': False,
 'is_kaggle': False,
 'p2config': Path('/home/vtec/.ecutilities/ecutilities.cfg'),
 'package_root': Path('/home/vtec/projects/ec-packages/eccore')}

In [ ]:
#| hide
machine.deregister_as_local();

In [ ]:
machine.is_local, machine.is_colab, machine.is_kaggle

(False, False, False)

This machine is not registered a local machine, but is also not running in the cloud. We should register it as a local machine with `register_as_local`

In [ ]:
show_doc(CurrentMachine.register_as_local)

---

[source](https://github.com/vtecftwy/eccore/blob/main/eccore/core.py#L229){target="_blank" style="float:right; font-size:smaller"}

### CurrentMachine.register_as_local

```python
def register_as_local():
```

*Update the configuration file to register the machine as local machine*

Use this method to register the current machine as local machine. Only needs to be used once on a machine. Do not use on cloud VMs

In [ ]:
machine.register_as_local()
machine.is_local, machine.is_colab, machine.is_kaggle

(True, False, False)

> **Technical Note**:
>
> The configuration file is located at a standard location, which varies depending on the OS:
> 
> - Windows:
>    - home is `C:\Users\username`
>    - application data in `C:\Users\username\AppData/Local/...` or `C:\Users\username\AppData\Roaming\...` (see [StackExchange](https://superuser.com/questions/21458/why-are-there-directories-called-local-locallow-and-roaming-under-users-user))
>    - application also can be loaded under a dedicated directory under `C:\Users\username` like `C:\Users\username\.conda\...`
>
> - Linux:
>     - home is `/home/username`
>     - application data in a file or dedicated directory `/home/username/` s.a.:
>         - file in home directory, e.g. `.gitconfig`
>         - file in an application dedicated directory, e.g. `/home/username/.conda/...`
> 
> `ecutilities` places the configuration file in a dedicated directory in the home directory:
> - `C:\Users\username\.ecutilities\ecutilities.cfg`
> - `/home/username/.ecutilities/ecutilities.cfg`
> 
> 
> Retrieve the OS:
> ```python
> sys.platform
> ```
> ```shell
> win32           with Windows
> linux           with linux
> darwin          with macOs
> ```
> 
> Accessing the correct path depending on the OS:
> ```python
> Path().home().absolute()
> ```
> ```shell
> WindowsPath('C:/Users/username') with Windows
> Path('/home/username')           with linux
> ``` 
> 

In [ ]:
#| export
class ProjectFileSystem(CurrentMachine):
    """Class representing the project file system and key subfolders (data, nbs, src)
    
    Set paths to key directories, according to whether the code is running locally or in the cloud.
    Give access to path to these key folders and information about the environment.
    """

    _instance = None
    _config_dir = '.ecutilities'
    _config_fname = 'ecutilities.cfg'
    _shared_project_dir = None
    
    def create_project_file_system(
        self, 
        p2project_root,     # path to project root, where all subfolder will be located
        overwrite=False     # overwrite current folders if they exist when True (not implemented yet)
        ):
        """Create a standard project file system with the following structure:
        
        ```
            project_root
                |--- data   all data files
                |--- nbs    all notebooks for work and experiments
                |--- src    all scripts and code
        ```
        """
        template = 'data nbs src'.split()
        path = safe_path(p2project_root)
        os.makedirs(path, exist_ok=True)
        for subdir in template:
            print(path/subdir)
            os.makedirs(path/subdir, exist_ok=True)
        print(f"Created project file system in {path}")

    @property
    def project_root(self):

        # TODO: this code is not correct. It only works when installed in the same folder as the project.
        
        if self.is_local:
            return PACKAGE_ROOT
        elif self.is_colab:
            return self.gdrive / self._shared_project_dir
        elif self.is_kaggle:
            raise NotImplemented(f"ProjectFileSystem is not implemented for Kaggle yet")
        else:
            raise ValueError('Not running locally, on Colab or on Kaggle')

    @property
    def data(self): return self.project_root / 'data'

    @property
    def nbs(self): return self.project_root / 'nbs'        

In [ ]:
pfs = ProjectFileSystem()
pfs()

{'os': 'linux',
 'home': Path('/home/vtec'),
 'is_local': True,
 'is_colab': False,
 'is_kaggle': False,
 'p2config': Path('/home/vtec/.ecutilities/ecutilities.cfg'),
 'package_root': Path('/home/vtec/projects/ec-packages/eccore')}

In [ ]:
show_doc(ProjectFileSystem.create_project_file_system)

---

[source](https://github.com/vtecftwy/eccore/blob/main/eccore/core.py#L276){target="_blank" style="float:right; font-size:smaller"}

### ProjectFileSystem.create_project_file_system

```python
def create_project_file_system(
    p2project_root, # path to project root, where all subfolder will be located
    overwrite:bool=False, # overwrite current folders if they exist when True (not implemented yet)
):
```

*Create a standard project file system with the following structure:*

```
    project_root
        |--- data   all data files
        |--- nbs    all notebooks for work and experiments
        |--- src    all scripts and code
```

In [ ]:
#|eval: false
pfs.create_project_file_system(Path('/home/vtec/projects/ec-packages/eccore'))

/home/vtec/projects/ec-packages/eccore/data
/home/vtec/projects/ec-packages/eccore/nbs
/home/vtec/projects/ec-packages/eccore/src
Created project file system in /home/vtec/projects/ec-packages/eccore


# Logging setup and functions

In [ ]:
#| export
def setup_logging(logfile:Path|None=None):
    """Setup logging to console and to file if logfile is not None"""

    # Setup logging file
    if logfile is None:
        print(f"No logfile provided. Logging to console only")
    else:
        print(f"Logging to console and to {logfile.absolute()}.")
        if not logfile.is_file():
            logfile.touch()

    # Configure the root logger
    root_logger = logging.getLogger()
    root_logger.setLevel(logging.DEBUG) # Set the root logger to catpure all levels of logs

    # Create a formatter
    formatter = logging.Formatter('%(asctime)s: %(message)s', datefmt='%Y-%m-%d %H:%M:%S')

    # Create a stream handler for console output
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.WARNING)  # Set the log level for the console handler
    console_handler.setFormatter(formatter)
    # Add console handler to the root logger
    root_logger.addHandler(console_handler)

    if logfile is not None:
        # Create a file handler to log to a file
        file_handler = logging.FileHandler(filename=logfile, mode='a', encoding='utf-8')
        file_handler.setLevel(logging.DEBUG)  # Set the log level for the file handler
        file_handler.setFormatter(formatter)
        # Add the file handler the root logger
        root_logger.addHandler(file_handler)
   
    # Custom exception handler to log uncaught exceptions at run time
    def handle_uncaught_exception(exc_type, exc_value, exc_traceback) -> None:
        if issubclass(exc_type, KeyboardInterrupt):
            sys.__excepthook__(exc_type, exc_value, exc_traceback)
            return root_logger.error("Uncaught exception", exc_info=(exc_type, exc_value, exc_traceback))

    # Set the custom exception handler
    sys.excepthook = handle_uncaught_exception

    print('Logging setup finished')


In [ ]:
#| export
def logthis(*args) -> None:
    """Logs all elements passed to logs"""
    text = ' '.join([str(element) for element in args])
    logging.info(text)

In [ ]:
#| export
def monitor_fn(fn):
    """Highlights when function in entered to and exited from"""
    @wraps(fn)
    def wrapper(*args, **kwargs):
        logthis(f"Entering `{fn.__name__}`")
        res = fn(*args, **kwargs)
        logthis(f"Exiting  `{fn.__name__}`")
        return res
    return wrapper


After setting up the logging, it is easy to create log entries:

In [ ]:
p2log = pfs.package_root / 'nbs-dev/data-dev/dev.log'
setup_logging(p2log)

Logging to console and to /home/vtec/projects/ec-packages/eccore/nbs-dev/data-dev/dev.log.
Logging setup finished


In [ ]:
logging.info('Logging manually as info, only shows in logfile')
logging.warning('Logging manually as warning, shows in logfile and console')

logthis('Using log function, as info, only shows in the log file')

2026-07-24 10:34:27: Logging manually as warning, shows in logfile and console


See logfile content:

In [ ]:
if p2log.exists():
    print('Log file content:')
    with open(p2log, 'r') as f:
        print(''.join(f.readlines()))

Log file content:
2026-07-24 10:34:27: Logging manually as info, only shows in logfile
2026-07-24 10:34:27: Logging manually as warning, shows in logfile and console
2026-07-24 10:34:27: Using log function, as info, only shows in the log file



Create a function decorated with `@monitor_fn` to monitor function calls, i.e. when function in entered and exited.

In [ ]:
@monitor_fn
def a_function(a,b):
    """Test functions to add two numbers"""
    return a + b

print(f"function output is {a_function(1,2)}")
print(f"")

function output is 3



In [ ]:
if p2log.exists():
    print('Log file content:')
    with open(p2log, 'r') as f:
        print(''.join(f.readlines()))

    p2log.unlink()

Log file content:
2026-07-24 10:34:27: Logging manually as info, only shows in logfile
2026-07-24 10:34:27: Logging manually as warning, shows in logfile and console
2026-07-24 10:34:27: Using log function, as info, only shows in the log file
2026-07-24 10:34:30: Entering `a_function`
2026-07-24 10:34:30: Exiting  `a_function`



# File structure exploration

In [ ]:
#| export
def files_in_tree(
    path: str|Path,               # path to the directory to scan  
    pattern: str|None = None      # pattern (glob style) to match in file name to filter the content
):
    """List files in directory and its subdiretories, print tree starting from parent directory"""
    validate_path(path, path_type='dir', raise_error=True)
    path = safe_path(path)
    pattern = '*' if pattern is None else f"*{pattern}*"
    parents = [p.name for p in path.parents]
    paths = []
    pad = ' ' * 2
    idx = 0
    print(f"{parents[0]}")
    print(f"{pad}|--{path.name}")
    for f in [p for p in path.glob(pattern) if p.is_file()]:
        paths.append(f)
        print(f"{pad}|{pad*2}|--{f.name} ({idx})")
        idx += 1
    for d in [p for p in path.iterdir() if p.is_dir()]:
        print(f"{pad}|{pad*2}|--{d.name}")
        for f in [p for p in d.glob(pattern) if p.is_file()]:
            paths.append(f)
            print(f"{pad}|{pad*2}|{pad*2}|--{f.name} ({idx})")
            idx += 1
    return paths

In [ ]:
show_doc(files_in_tree)

---

[source](https://github.com/vtecftwy/eccore/blob/main/eccore/core.py#L383){target="_blank" style="float:right; font-size:smaller"}

### files_in_tree

```python
def files_in_tree(
    path:str | Path, # path to the directory to scan
    pattern:str | None=None, # pattern (glob style) to match in file name to filter the content
):
```

*List files in directory and its subdiretories, print tree starting from parent directory*

In [ ]:
p2dir = Path('').resolve()
print(p2dir, '\n')

files = files_in_tree(p2dir)
print(f"List of {len(files)} files when unfiltered")

/home/vtec/projects/ec-packages/eccore/nbs-dev 

eccore
  |--nbs-dev
  |    |--index.ipynb (0)
  |    |--0_02_plotting.ipynb (1)
  |    |--styles.css (2)
  |    |--9_01_dev_utils.ipynb (3)
  |    |--nbdev.yml (4)
  |    |--_quarto.yml (5)
  |    |--0_00_core.ipynb (6)
  |    |--0_01_ipython.ipynb (7)
  |    |--sidebar.yml (8)
  |    |--.last_checked (9)
  |    |--data-dev
  |    |    |--ten-blobs-6-cols-y.npy (10)
  |    |    |--ten-blobs-6-cols-X.npy (11)
  |    |    |--jsondict-test.json (12)
  |    |    |--ten-blobs-6-cols-clusters.npy (13)
List of 14 files when unfiltered


Use `pattern` to filter the paths to return (using `glob` syntax)

In [ ]:
files = files_in_tree(p2dir, pattern='ipynb')
print(f"List of {len(files)} files when filtered")

eccore
  |--nbs-dev
  |    |--index.ipynb (0)
  |    |--0_02_plotting.ipynb (1)
  |    |--9_01_dev_utils.ipynb (2)
  |    |--0_00_core.ipynb (3)
  |    |--0_01_ipython.ipynb (4)
  |    |--data-dev
List of 5 files when filtered


In [ ]:
#| export
def path_to_parent_dir(
    pattern:str,               # pattern to identify the parent directory
    path:str|Path|None = None, # optional path from where to seek for parent directory
)-> Path:                      # path of the parent directory
    """Climb directory tree up to a directory starting with `pattern`, and return its path.
    
    - When no directory is found in the tree starting with `pattern`, return the current directory path.
    
    - It is possible to pass a `path` as starting path to climb from. 
    """
    if path is None: path = Path()
    path = safe_path(path).absolute()
    tree = [path.name] + [p.name for p in path.parents]
    mask = [True if n.startswith(pattern) else False for n in tree]
    # A parent directory with the pattern is found in the tree, return that directory
    if any(mask):
        tree = tree[mask.index(True):]
        tree.reverse()
        parent_dir = Path('/'.join(tree))
    # No parent directory with the pattern is found in the tree, return the current directory
    else:
        parent_dir = Path().absolute()
    return parent_dir

In [ ]:
show_doc(path_to_parent_dir)

---

[source](https://github.com/vtecftwy/eccore/blob/main/eccore/core.py#L410){target="_blank" style="float:right; font-size:smaller"}

### path_to_parent_dir

```python
def path_to_parent_dir(
    pattern:str, # pattern to identify the parent directory
    path:str | Path | None=None, # optional path from where to seek for parent directory
)->Path: # path of the parent directory
```

*Climb directory tree up to a directory starting with `pattern`, and return its path.*

- When no directory is found in the tree starting with `pattern`, return the current directory path.

- It is possible to pass a `path` as starting path to climb from.

In [ ]:
p2dir = path_to_parent_dir('nbs')
assert 'nbs-dev' in p2dir.parts and 'nbs' not in p2dir.parts
p2dir

Path('/home/vtec/projects/ec-packages/eccore/nbs-dev')

In [ ]:
# p2dir = path_to_parent_dir('nbs', Path('../nbs/sandbox.ipynb').resolve())
# assert 'nbs' in p2dir.parts and 'nbs-dev' not in p2dir.parts
# p2dir

In [ ]:
p2dir = path_to_parent_dir('not-in-tree').resolve()
assert p2dir == Path().absolute()

In [ ]:
#|eval: false
p2project_root = path_to_parent_dir('eccore')
assert 'eccore' in p2project_root.parts and 'nbs' not in p2project_root.parts
p2project_root

Path('/home/vtec/projects/ec-packages/eccore')

In [ ]:
#| hide
nbdev_export()